In [1]:
explanation_memory = {
    "intent": None,
    "confidence": None,
    "keywords": []
}
# =========================
# KNOWLEDGE BASE
# =========================

CONTACT_INFO = {
    "phone": "061-430000 / 061-435000",
    "email": "democontact@pension.com.np",
    "whatsapp": "+977 9700000000",
    "address": "Pokhara, Nepal"
}

DOCUMENTS = {
    "new_pension": [
        "Citizenship ID",
        "Pension bank account (NSBL/EBL)",
        "Passport photos",
        "Discharge certificate"
    ],
    "family_pension": [
        "Death certificate",
        "Citizenship of family member",
        "Family details certificate",
        "Photos"
    ],
    "lta": [
        "Death certificate",
        "Relation certificate",
        "Citizenship",
        "Pension certificate",
        "2 witnesses"
    ]
}

# =========================
# INTENTS
# =========================

INTENTS = {
    "greeting": ["hi", "hello", "hey"],
    "howareyou": ["how are you"],
    "contact": ["contact", "call", "phone", "email", "address"],
    "payment": ["pension", "paid", "received", "not received", "delay", "missing"],
    "documents": ["document", "papers", "required"],
    "widow": ["widow", "family pension"],
    "echs": ["echs"],
    "timings": ["open", "time", "hours"],
    "renewal": ["life certificate", "renew"],
    "lta": ["lta"],
}

# =========================
# CONTEXT (memory)
# =========================

context = {
    "last_intent": None
}

# =========================
# INTENT DETECTION
# =========================

def detect_intent(user):
    text = user.lower()
    scores = {}
    keyword_hits = {}

    for intent, keywords in INTENTS.items():
        hits = [word for word in keywords if word in text]
        score = len(hits)

        scores[intent] = score
        keyword_hits[intent] = hits

    best_intent = max(scores, key=scores.get)
    best_score = scores[best_intent]

    if best_score >= 2:
        confidence = 0.8
    elif best_score == 1:
        confidence = 0.5
    else:
        confidence = 0

    # Store explanation data
    explanation_memory["intent"] = best_intent
    explanation_memory["confidence"] = confidence
    explanation_memory["keywords"] = keyword_hits[best_intent]

    return best_intent, confidence


# =========================
# RESPONSE LOGIC
# =========================

def get_response(intent, user):
    text = user.lower()

    # GREETING
    if intent == "greeting":
        return "Hello! 👋 How can I assist you with pension services today?"

    if intent == "howareyou":
        return "I'm doing well 😊 I'm here to help you with any pension-related questions."

    # CONTACT
    if intent == "contact":
        return (
            f"You can contact the Pension Office here:\n"
            f"📍 {CONTACT_INFO['address']}\n"
            f"📞 {CONTACT_INFO['phone']}\n"
            f"📧 {CONTACT_INFO['email']}\n"
            f"💬 WhatsApp: {CONTACT_INFO['whatsapp']}"
        )

    # PAYMENT
    if intent == "payment":
        if "not" in text or "missing" in text:
            return (
                "It seems your pension has not been received.\n"
                f"Please contact the office at {CONTACT_INFO['phone']} or WhatsApp {CONTACT_INFO['whatsapp']}."
            )
        return "Pension is usually credited monthly after processing is complete."

    # DOCUMENTS (BRANCHING)
    if intent == "documents":
        context["last_intent"] = "documents"
        return (
            "Which documents do you need?\n"
            "1. New Pension\n"
            "2. Family/Widow Pension\n"
            "3. LTA\n"
            "Type your choice."
        )

    # HANDLE DOCUMENT FOLLOW-UP
    if context["last_intent"] == "documents":
        if "1" in text:
            return "Required documents for new pension:\n- " + "\n- ".join(DOCUMENTS["new_pension"])
        elif "2" in text:
            return "Required documents for family pension:\n- " + "\n- ".join(DOCUMENTS["family_pension"])
        elif "3" in text:
            return "Required documents for LTA:\n- " + "\n- ".join(DOCUMENTS["lta"])

    # WIDOW
    if intent == "widow":
        return "For widow pension, please bring death certificate, ID, and service records."

    # ECHS
    if intent == "echs":
        return "For ECHS membership, please visit the ECHS Office (MI Room) for registration."

    # TIMINGS
    if intent == "timings":
        return "The office is open Monday to Friday, 9 AM to 5 PM."

    # RENEWAL
    if intent == "renewal":
        return "Life certificate must be submitted yearly between September and November."

    # LTA
    if intent == "lta":
        return "LTA requires documents like death certificate, relation proof, and pension certificate."

    return "I'm not sure I understood that 🤔 You can ask about pension, documents, or contact help."


def explain_response():
    if explanation_memory["intent"] is None:
        return "I haven't processed a request yet."

    intent = explanation_memory["intent"]
    confidence = explanation_memory["confidence"]
    keywords = explanation_memory["keywords"]

    return (
        f"I answered that because I detected the intent '{intent}'.\n"
        f"Matched keywords: {', '.join(keywords) if keywords else 'none'}.\n"
        f"Confidence level: {confidence}.\n"
        "I used rule-based matching to decide the most relevant response."
    )

# =========================
# MAIN BOT
# =========================

def pension_bot(user):
    if any(x in user.lower() for x in ["why", "explain"]):
        return explain_response()
        
    intent, confidence = detect_intent(user)

    if confidence == 0:
        return "Hmm, I didn’t quite understand that. Could you rephrase?"

    response = get_response(intent, user)
    context["last_intent"] = intent

    return response


# =========================
# CHAT LOOP
# =========================


In [ ]:
while True:
    user = input("You: ")

    if user.lower() == "exit":
        print("Bot: Goodbye! Take care 👋")
        break

    print("Bot:", pension_bot(user))

You:  hi


Bot: Hello! 👋 How can I assist you with pension services today?


You:  explain


Bot: I answered that because I detected the intent 'greeting'.
Matched keywords: hi.
Confidence level: 0.5.
I used rule-based matching to decide the most relevant response.


You:  how are you


Bot: I'm doing well 😊 I'm here to help you with any pension-related questions.


You:  i want help with pensions and stuff


Bot: Pension is usually credited monthly after processing is complete.


You:  how do i contact the office


Bot: You can contact the Pension Office here:
📍 Rambazar-15, Pokhara, Nepal
📞 061-435804 / 061-435805
📧 oicpok.pokhara@mea.gov.in
💬 WhatsApp: +977 9765651149
